# Example: Frequency-Dependent Resolution

This example demonstrates `sid.freq_btfdr`, which uses a different window
size at each frequency. This is valuable when a system has sharp features
(e.g., resonances) that need fine resolution at some frequencies but not
others. Replaces MATLAB's `spafdr`.

Translated from `exampleFreqDepRes.m`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
import sid

%matplotlib inline

## Generate test data: second-order resonant system

Poles at $0.9\,e^{\pm j\pi/4}$ create a resonance peak near $\omega = \pi/4$.

In [ ]:
rng = np.random.default_rng(2)

N = 5000
Ts = 0.01
b = [1]
a_coeff = [1, -2 * 0.9 * np.cos(np.pi / 4), 0.9**2]
u = rng.standard_normal(N)
y = lfilter(b, a_coeff, u) + 0.1 * rng.standard_normal(N)

## Fixed-window `freq_bt`: the resolution-variance trade-off

Small M (=15): smooth but misses the resonance peak.
Large M (=80): captures the peak but has high variance.

In [ ]:
r_small = sid.freq_bt(y, u, window_size=15, sample_time=Ts)
r_large = sid.freq_bt(y, u, window_size=80, sample_time=Ts)

w = r_small.frequency
G_true = 1.0 / (1.0 - 2 * 0.9 * np.cos(np.pi / 4) * np.exp(-1j * w)
               + 0.81 * np.exp(-2j * w))

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(w, 20 * np.log10(np.abs(r_small.response)), 'b', label='BT M=15')
ax.semilogx(w, 20 * np.log10(np.abs(r_large.response)), 'r', label='BT M=80')
ax.semilogx(w, 20 * np.log10(np.abs(G_true)), 'k--', lw=1.5, label='True')
ax.set_xlabel('Frequency (rad/sample)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('Fixed Window: Resolution vs Variance Trade-off')
ax.legend(loc='lower left')
ax.grid(True)
plt.tight_layout()
plt.show()

## Scalar resolution with `freq_btfdr`

Resolution R sets the window size as $M = \lceil 2\pi / R \rceil$.
Smaller R = finer resolution (larger window).

In [ ]:
result_fdr = sid.freq_btfdr(y, u, resolution=0.2, sample_time=Ts)

sid.bode_plot(result_fdr)
plt.suptitle('freq_btfdr with Scalar Resolution R = 0.2', y=1.02)
plt.tight_layout()
plt.show()

## Per-frequency resolution vector

Use fine resolution near the resonance (low frequencies) and coarse
resolution at high frequencies where the response is flat.

In [ ]:
nf = len(result_fdr.frequency)
R_vec = np.linspace(0.1, 1.5, nf)   # fine at low freq, coarse at high freq

result_vec = sid.freq_btfdr(y, u, resolution=R_vec, sample_time=Ts)

# The resulting window_size varies across frequency
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7))

ax1.plot(result_vec.frequency, result_vec.window_size, 'b')
ax1.set_xlabel('Frequency (rad/sample)')
ax1.set_ylabel('Window Size M')
ax1.set_title('Per-Frequency Window Size')
ax1.grid(True)

ax2.semilogx(result_vec.frequency, 20 * np.log10(np.abs(result_vec.response)),
             'b', label='BTFDR (variable R)')
ax2.semilogx(w, 20 * np.log10(np.abs(G_true)), 'k--', lw=1.5, label='True')
ax2.set_xlabel('Frequency (rad/sample)')
ax2.set_ylabel('Magnitude (dB)')
ax2.set_title('BTFDR with Per-Frequency Resolution')
ax2.legend(loc='lower left')
ax2.grid(True)

plt.tight_layout()
plt.show()

## Compare BT vs BTFDR side by side

BTFDR adapts to capture the peak while keeping variance low elsewhere.

In [ ]:
r_bt  = sid.freq_bt(y, u, window_size=30, sample_time=Ts)
r_fdr = sid.freq_btfdr(y, u, resolution=0.3, sample_time=Ts)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(r_bt.frequency, 20 * np.log10(np.abs(r_bt.response)),
            'b', label='BT (M=30)')
ax.semilogx(r_fdr.frequency, 20 * np.log10(np.abs(r_fdr.response)),
            'r', label='BTFDR (R=0.3)')
ax.semilogx(r_bt.frequency, 20 * np.log10(np.abs(G_true)),
            'k--', lw=1.5, label='True')
ax.set_xlabel('Frequency (rad/sample)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('Blackman-Tukey: Fixed vs Frequency-Dependent Resolution')
ax.legend(loc='lower left')
ax.grid(True)
plt.tight_layout()
plt.show()